# Compare GOF: Philipp vs Neha

## What is the GOF?

The goodness-of-fit (GOF) test answers: **does the best-fit model actually describe the data well, bin by bin?**

The fit minimises the negative log-likelihood $-\log L(\hat\theta)$ over the model parameters $\theta$. The best-fit likelihood $L_\text{best}$ is the highest the model can achieve — but it still may not match the data perfectly, because the model is constrained to a specific functional form (e.g. a power-law spectrum).

The **saturated likelihood** $L_\text{sat}$ is the absolute maximum: the likelihood when the expected count in *every bin* equals the observed count ($\hat\mu_i = k_i$). It sets the ceiling — no model can do better.

The GOF test statistic is:
$$\mathrm{TS} = 2\log\frac{L_\mathrm{sat}}{L_\mathrm{best}} = 2\sum_i \left[\log p(k_i | k_i) - \log p(k_i | \hat\mu_i)\right] \geq 0$$

A small TS means the best fit is close to perfect; a large TS means the model shape is a poor description of the data.

**Under Wilks' theorem**, TS follows a $\chi^2$ distribution with $N_\text{dof} = N_\text{bins} - N_\text{params}$ degrees of freedom, giving an analytic p-value. In practice Wilks may not hold exactly (boundary conditions, low statistics, non-Gaussian LLH), so we validate by generating **pseudoexperiments**: Poisson-fluctuate fake datasets from the best-fit model, refit each, compute their TS, and take the fraction exceeding the data TS as the empirical p-value.

---

Overlays and compares the GOF implementations from:
- **Philipp**: `step0_5_eval_gof_pseudoexp.ipynb` — data TS from `do_scan_analysis.py`
- **Neha**: `step1_hese_flux/GOF.ipynb` — data TS from `utils/goodness_of_fit.py::calculate_TS_datafit`

## How Philipp's sat LLH is computed (`do_scan_analysis.py`)

```python
def calc_sat_llh(data_hist, calculation_type="poisson"):
    sat_llh = 0
    for det_config in data_hist:
        k = data_hist[det_config]
        sat_llh += np.sum(likelihoods.PoissonLLH.compute_log_L_non_graph(k, k))
    return -1 * sat_llh   # <-- minus sign: we save -llh in all cases
```

`PoissonLLH.compute_log_L_non_graph(k, k)` calls `scipy.stats.poisson.logpmf(k, k)` = $\log\mathrm{Poisson}(k|k) < 0$.

After `× −1`: **`sat_llh_yaml = −log(L_sat) > 0`** (stored in the YAML).

Data TS in the pseudoexp notebook:
$$\mathrm{TS} = 2\,(\texttt{min\_llh} - \texttt{sat\_llh\_yaml}) = 2\,(-\log L_\mathrm{best} - (-\log L_\mathrm{sat})) = 2\log\frac{L_\mathrm{sat}}{L_\mathrm{best}} \checkmark$$

## How Neha's sat LLH is computed (`goodness_of_fit.py`)

```python
def binwise_saturated_poisson_llh(data):
    return data * (-1. + np.log(data + 1E-300)) - loggamma(data + 1.0).real
```

This is $k(-1+\ln k) - \ln\Gamma(k+1) = k\ln k - k - \ln k! = \log\mathrm{Poisson}(k|k) < 0$ (same formula, tiny `1E-300` guard for $k=0$).

**`sat_llh_neha = +log(L_sat) < 0`** (not negated).

Data TS in Neha's code:
$$\mathrm{TS} = 2\,(\texttt{llh} + \texttt{sat\_llh\_neha}) = 2\,(-\log L_\mathrm{best} + \log L_\mathrm{sat}) = 2\log\frac{L_\mathrm{sat}}{L_\mathrm{best}} \checkmark$$

## Summary of all differences

| Aspect | Philipp | Neha |
|--------|---------|------|
| **sat_llh formula** | `scipy.poisson.logpmf(k,k)` × **−1** → positive | `k*(−1+log k) − loggamma(k+1)` → negative |
| **TS formula** | `2*(llh − sat_llh)` | `2*(llh + sat_llh)` |
| **Both reduce to** | $2\log(L_\mathrm{sat}/L_\mathrm{best})$ | $2\log(L_\mathrm{sat}/L_\mathrm{best})$ |
| **Data TS source** | YAML (precomputed by `do_scan_analysis.py`) | Pickle + NNMFitter, computed on the fly |
| **mask_unsuccessful** | `False` | `True` |
| **Chi2 reference** | Theoretical DOF = 579 (bins − params) | Chi2 fit to pseudoexp distribution |
| **Pseudoexp datasets** | LBFGSB + force_full_3 × round3 + seeded | LBFGSB / round3 only |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import yaml
from scipy.stats import chi2
from scipy.special import loggamma

from NNMFit.utilities import PseudoexpHandler
from NNMFit import likelihoods

sys.path.append("/data/user/nlad/NNMFitStuff")
from utils.goodness_of_fit import calculate_TS_datafit, binwise_saturated_poisson_llh
from utils.pseudoexp_handling import PseudoexpPlotHandler

## 1. Cross-check: are the two sat_llh formulas numerically identical?

In [ ]:
# Test on a range of bin counts including 0
k_test = np.array([0., 1., 2., 5., 10., 23., 100., 250.])

# Philipp's formula (before the -1): scipy.stats.poisson.logpmf(k, k)
philipp_logL = likelihoods.PoissonLLH.compute_log_L_non_graph(k_test, k_test)

# Neha's formula
neha_logL = binwise_saturated_poisson_llh(k_test)

print(f"{'k':>6}  {'Philipp logpmf(k,k)':>22}  {'Neha formula':>22}  {'diff':>12}")
print("-" * 70)
for k, p, n in zip(k_test, philipp_logL, neha_logL):
    print(f"{k:6.0f}  {p:22.10f}  {n:22.10f}  {p-n:12.2e}")

## 2. Load pseudoexperiments

In [ ]:
pse_path = '/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/pse'
minimizer = 'LBFGSB'
iteration  = 'pse_hese_spectrum_bestfit_13year_n500_round3'
indir = f'{pse_path}/{minimizer}/{iteration}'

# Philipp's handler: mask_unsuccessful=False, sat_llh pre-computed as -log(L_sat)
hdl_philipp = PseudoexpHandler(
    indir,
    force_read=True,
    mask_unsuccessful=False,
    sat_llh_calculation_type="poisson",
)

# Neha's handler: mask_unsuccessful=True
hdl_neha_inner = PseudoexpHandler(
    indir,
    force_read=True,
    mask_unsuccessful=True,
)

plot_hdl = PseudoexpPlotHandler()
plot_hdl.add_pseudoexp(
    identifier='HESE-13, batch1',
    hdl=hdl_neha_inner,
    label='Pseudoexps (Neha)',
    color='C1',
)

## 3. Compute pseudoexp TS: both methods

### Philipp
`PseudoexpHandler` stores `sat_llh = −log(L_sat) > 0` (same convention as `do_scan_analysis.py`).  
Formula: `ts = 2*(llh − sat_llh)`.

In [ ]:
df_philipp = hdl_philipp.get_pseudoexp_df()
ts_philipp = 2.0 * (df_philipp["llh"] - df_philipp["sat_llh"]).values

print(f"Philipp (mask_unsuccessful=False): {len(ts_philipp)} pseudoexps")
print(f"  median TS = {np.median(ts_philipp):.2f}")
print(f"  mean   TS = {np.mean(ts_philipp):.2f}")
print(f"  min/max   = {ts_philipp.min():.2f} / {ts_philipp.max():.2f}")

### Neha
`get_TS_values(llh_type='say', say_sat_llh_type='poisson')` iterates seeds, computes `sat_llh = sum(binwise_sat_poisson(data)) < 0`.  
Formula: `ts = 2*(llh + sat_llh)`.

In [ ]:
ts_neha = np.array(plot_hdl.get_TS_values(
    'HESE-13, batch1',
    llh_type='say',
    say_sat_llh_type='poisson',
))

print(f"Neha (mask_unsuccessful=True): {len(ts_neha)} pseudoexps")
print(f"  median TS = {np.median(ts_neha):.2f}")
print(f"  mean   TS = {np.mean(ts_neha):.2f}")
print(f"  min/max   = {ts_neha.min():.2f} / {ts_neha.max():.2f}")

## 4. Cross-check per-pseudoexp TS agreement

For seeds present in both (i.e., successful fits), the TS should be identical to floating-point precision.

In [ ]:
# Reload Philipp's handler with mask_unsuccessful=True so seeds match Neha's
hdl_philipp_masked = PseudoexpHandler(
    indir,
    force_read=True,
    mask_unsuccessful=True,
    sat_llh_calculation_type="poisson",
)
df_philipp_masked = hdl_philipp_masked.get_pseudoexp_df()
ts_philipp_masked = 2.0 * (df_philipp_masked["llh"] - df_philipp_masked["sat_llh"]).values

print(f"Philipp (masked, {len(ts_philipp_masked)} pseudoexps) vs Neha ({len(ts_neha)} pseudoexps)")
assert len(ts_philipp_masked) == len(ts_neha), "Seed count mismatch — check ordering!"

diff = ts_philipp_masked - ts_neha
print(f"Max |TS_Philipp − TS_Neha| = {np.max(np.abs(diff)):.6e}")
print(f"Mean |diff|                = {np.mean(np.abs(diff)):.6e}")

## 5. Compute data TS: both methods

### Philipp — from YAML written by `do_scan_analysis.py`
`TS = 2 * (min_llh − sat_llh_yaml)`  where `sat_llh_yaml = −log(L_sat) > 0`.

In [ ]:
llh_info_file = (
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/philipp/"
    "saturated/force_full_3_nlight_20/SatLLH_nbestfit20_scan_astro_gamma_1D_10steps_round2.txt"
)
with open(llh_info_file) as f:
    llh_info = yaml.safe_load(f)

ts_data_philipp = 2.0 * (llh_info["min LLH"] - llh_info["Saturated LLH"])

print("Philipp data TS (from YAML, produced by do_scan_analysis.py):")
print(f"  min LLH        = {llh_info['min LLH']:.6f}   (= -log L_best)")
print(f"  Saturated LLH  = {llh_info['Saturated LLH']:.6f}   (= -log L_sat, via -1*logpmf(k,k))")
print(f"  TS             = 2*({llh_info['min LLH']:.4f} - {llh_info['Saturated LLH']:.4f}) = {ts_data_philipp:.4f}")

### Neha — computed on the fly from the pickle file
`calculate_TS_datafit` loads the NNMFitter from the config, reads real data hists, and computes `TS = 2*(llh + Σ binwise_sat_poisson(k))`.

In [ ]:
data_fit_file = (
    '/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/'
    'hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/'
    'nbestfit20_scan_astro_gamma_1D_10steps_round2/Freefit_17.pickle'
)

ts_data_neha = calculate_TS_datafit(
    fit_res_file=data_fit_file,
    llh_type='say',
    say_sat_llh_type='poisson',
)
print(f"Neha data TS (from pickle + NNMFitter): {ts_data_neha:.6f}")

In [ ]:
print("Data TS comparison:")
print(f"  Philipp (YAML / do_scan_analysis.py) : {ts_data_philipp:.6f}")
print(f"  Neha   (NNMFitter on-the-fly)        : {ts_data_neha:.6f}")
print(f"  Difference                           : {ts_data_philipp - ts_data_neha:.6e}")

## 6. Overlay TS distributions

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

all_ts = np.concatenate([ts_philipp, ts_neha])
bins = np.linspace(np.percentile(all_ts, 0.5), np.percentile(all_ts, 99.5), 35)

# Philipp (unmasked)
hist_p, _ = np.histogram(ts_philipp, bins=bins, density=True)
ax.stairs(hist_p, bins, color='C0', linewidth=1.5,
          label=f'Philipp  (n={len(ts_philipp)}, mask=False)')

# Neha (masked)
hist_n, _ = np.histogram(ts_neha, bins=bins, density=True)
ax.stairs(hist_n, bins, color='C1', linewidth=1.5, linestyle='--',
          label=f'Neha     (n={len(ts_neha)}, mask=True)')

# Data TS
ax.axvline(ts_data_philipp, color='C0', linestyle='-.',
           label=f'data TS Philipp (YAML) = {ts_data_philipp:.2f}')
ax.axvline(ts_data_neha, color='C1', linestyle=':',
           label=f'data TS Neha (NNMFitter) = {ts_data_neha:.2f}')

# Philipp's reference chi2: theoretical DOF
dof_philipp = 579  # (23*10 + 23*10 + 13*10) - 11
x_chi2 = np.linspace(bins[0], bins[-1], 300)
ax.plot(x_chi2, chi2.pdf(x_chi2, df=dof_philipp),
        color='grey', linewidth=1.2,
        label=rf'$\chi^2$(dof={dof_philipp}) — Philipp analytic')

# Neha's reference chi2: fitted to pseudoexps
chi2_fit = chi2.fit(ts_neha, floc=0., fscale=1.)
ax.plot(x_chi2, chi2.pdf(x_chi2, df=chi2_fit[0]),
        color='orange', linewidth=1.2, linestyle='--',
        label=rf'$\chi^2$(dof={chi2_fit[0]:.1f}) fit to Neha pseudoexps')

ax.set_xlabel(
    r'$\mathrm{TS} = 2\log(L_\mathrm{sat}/L_\mathrm{best})$',
    fontsize=13
)
ax.set_ylabel('pdf', fontsize=13)
ax.set_title('GOF: Philipp vs Neha — LBFGSB / round3', fontsize=12)
ax.legend(fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.show()

## 7. Effect of mask_unsuccessful on TS distribution

Isolate this difference by using Philipp's TS formula with both mask settings.

In [ ]:
n_total   = len(ts_philipp)
n_masked  = len(ts_philipp_masked)
n_removed = n_total - n_masked

print(f"Total pseudoexps (unmasked) : {n_total}")
print(f"After mask_unsuccessful=True: {n_masked}")
print(f"Removed as unsuccessful      : {n_removed} ({100*n_removed/n_total:.1f}%)")

fig, ax = plt.subplots(figsize=(7, 4), dpi=130)
ax.stairs(*np.histogram(ts_philipp,        bins=bins, density=True)[::-1][::-1],
          color='C0', label=f'mask=False (n={n_total})')
ax.stairs(*np.histogram(ts_philipp_masked, bins=bins, density=True)[::-1][::-1],
          color='C2', linestyle='--', label=f'mask=True  (n={n_masked})')
ax.set_xlabel(r'TS', fontsize=12)
ax.set_ylabel('pdf', fontsize=12)
ax.set_title('Effect of mask_unsuccessful (Philipp TS formula)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

## 8. p-values

In [ ]:
ts_data = ts_data_philipp  # both methods should agree

p_analytic   = chi2.sf(ts_data, df=dof_philipp)
p_chi2fit    = chi2.sf(ts_data, df=chi2_fit[0])
p_emp_philipp = np.mean(ts_philipp        >= ts_data)
p_emp_neha    = np.mean(ts_neha           >= ts_data)

print(f"Data TS = {ts_data:.2f}\n")
print(f"  chi2(dof=579) analytic (Philipp)     : p = {p_analytic:.4f}  ({p_analytic*100:.2f}%)")
print(f"  chi2(dof={chi2_fit[0]:.1f}) fitted (Neha)      : p = {p_chi2fit:.4f}  ({p_chi2fit*100:.2f}%)")
print(f"  empirical, unmasked (Philipp pse)    : p = {p_emp_philipp:.4f}  ({p_emp_philipp*100:.2f}%)")
print(f"  empirical, masked   (Neha pse)       : p = {p_emp_neha:.4f}  ({p_emp_neha*100:.2f}%)")